# 01 — S3: Data Lake Foundations

S3 is the core storage layer for almost every AWS data engineering pipeline. This notebook covers:

- Section A: Bucket lifecycle (create, tag, version, list, delete)
- Section B: Object operations (put, get, list, copy, delete)
- Section C: Versioning & lifecycle rules
- Section D: Hive-style partition layout (year/month/day)

In [1]:
import sys, io, json
sys.path.insert(0, "..")
import helpers  # sets dummy credentials automatically

import boto3
import pandas as pd
from moto import mock_aws

---
## Section A — Bucket Lifecycle

In [2]:
# A1: Create a bucket
# GOTCHA: us-east-1 does NOT accept CreateBucketConfiguration — all other regions do.
with mock_aws():
    s3 = helpers.make_boto3_client("s3")

    # us-east-1: no location constraint
    s3.create_bucket(Bucket="my-datalake-use1")

    # Any other region: must pass CreateBucketConfiguration
    s3.create_bucket(
        Bucket="my-datalake-eu",
        CreateBucketConfiguration={"LocationConstraint": "eu-west-1"},
    )

    buckets = [b["Name"] for b in s3.list_buckets()["Buckets"]]
    print("Buckets created:", buckets)

Buckets created: ['my-datalake-use1', 'my-datalake-eu']


In [3]:
# A2: Add tags (cost allocation tags are important in real DE environments)
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="tagged-lake")
    s3.put_bucket_tagging(
        Bucket="tagged-lake",
        Tagging={"TagSet": [
            {"Key": "Team", "Value": "DataEngineering"},
            {"Key": "CostCenter", "Value": "DE-001"},
        ]},
    )
    tags = s3.get_bucket_tagging(Bucket="tagged-lake")["TagSet"]
    print("Tags:", tags)

Tags: [{'Key': 'Team', 'Value': 'DataEngineering'}, {'Key': 'CostCenter', 'Value': 'DE-001'}]


In [4]:
# A3: Get bucket location
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="location-demo")
    loc = s3.get_bucket_location(Bucket="location-demo")
    # us-east-1 returns None for LocationConstraint
    print("Location:", loc["LocationConstraint"])

Location: None


---
## Section B — Object Operations

In [5]:
# B1: put_object and get_object
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="objects-demo")

    # Upload raw bytes
    s3.put_object(Bucket="objects-demo", Key="raw/hello.txt", Body=b"Hello, Data Lake!")

    # Upload a string (must encode to bytes)
    s3.put_object(Bucket="objects-demo", Key="raw/data.csv", Body="id,name\n1,Alice\n2,Bob".encode())

    # Get object — Body is a StreamingBody, must call .read()
    resp = s3.get_object(Bucket="objects-demo", Key="raw/hello.txt")
    text = resp["Body"].read().decode()
    print("Downloaded:", text)

Downloaded: Hello, Data Lake!


In [6]:
# B2: Upload a DataFrame as CSV using helper
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="df-demo")

    df = pd.DataFrame({"id": [1, 2, 3], "city": ["Bangkok", "Singapore", "Tokyo"]})
    helpers.upload_df_as_csv(s3, df, "df-demo", "cities/data.csv")

    # Read it back
    df_back = helpers.download_df_from_csv(s3, "df-demo", "cities/data.csv")
    print(df_back)

   id       city
0   1    Bangkok
1   2  Singapore
2   3      Tokyo


In [7]:
# B3: Upload and download Parquet
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="parquet-demo")

    df = pd.DataFrame({"order_id": range(5), "amount": [10.5, 20.0, 15.3, 8.9, 32.1]})
    helpers.upload_df_as_parquet(s3, df, "parquet-demo", "orders/2024-01.parquet")

    df_back = helpers.download_df_from_parquet(s3, "parquet-demo", "orders/2024-01.parquet")
    print(df_back)
    assert df_back.shape == (5, 2), "Shape mismatch!"
    print("Parquet round-trip: OK")

   order_id  amount
0         0    10.5
1         1    20.0
2         2    15.3
3         3     8.9
4         4    32.1
Parquet round-trip: OK


In [8]:
# B4: list_objects_v2 with pagination
# list_objects_v2 returns at most 1000 objects per call.
# For more, loop using NextContinuationToken.
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="paginate-demo")

    # Upload 5 objects
    for i in range(5):
        s3.put_object(Bucket="paginate-demo", Key=f"raw/file_{i:04d}.csv", Body=b"data")

    # ---- Pagination pattern (interview must-know) ----
    all_keys = []
    kwargs = {"Bucket": "paginate-demo", "Prefix": "raw/"}
    while True:
        resp = s3.list_objects_v2(**kwargs)
        all_keys.extend(obj["Key"] for obj in resp.get("Contents", []))
        if resp.get("IsTruncated"):               # more pages exist
            kwargs["ContinuationToken"] = resp["NextContinuationToken"]
        else:
            break

    print(f"Found {len(all_keys)} objects:")
    for k in all_keys:
        print(" ", k)

# helpers.list_all_objects() wraps this pattern for convenience

Found 5 objects:
  raw/file_0000.csv
  raw/file_0001.csv
  raw/file_0002.csv
  raw/file_0003.csv
  raw/file_0004.csv


In [9]:
# B5: copy_object — move data between prefixes WITHOUT downloading
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="copy-demo")
    s3.put_object(Bucket="copy-demo", Key="raw/file.csv", Body=b"id,val\n1,100")

    # Copy from raw/ to processed/
    s3.copy_object(
        Bucket="copy-demo",
        Key="processed/file.csv",
        CopySource={"Bucket": "copy-demo", "Key": "raw/file.csv"},
    )

    keys = [o["Key"] for o in helpers.list_all_objects(s3, "copy-demo")]
    print("Objects after copy:", keys)

Objects after copy: ['processed/file.csv', 'raw/file.csv']


In [10]:
# B6: delete_object and batch delete_objects
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="delete-demo")
    for i in range(4):
        s3.put_object(Bucket="delete-demo", Key=f"file_{i}.txt", Body=b"x")

    # Single delete
    s3.delete_object(Bucket="delete-demo", Key="file_0.txt")

    # Batch delete (up to 1000 at a time)
    s3.delete_objects(
        Bucket="delete-demo",
        Delete={"Objects": [{"Key": "file_1.txt"}, {"Key": "file_2.txt"}]},
    )

    remaining = [o["Key"] for o in helpers.list_all_objects(s3, "delete-demo")]
    print("Remaining objects:", remaining)  # only file_3.txt

Remaining objects: ['file_3.txt']


---
## Section C — Versioning & Lifecycle Rules

In [11]:
# C1: Enable versioning and inspect versions
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="versioned-lake")

    # Enable versioning
    s3.put_bucket_versioning(
        Bucket="versioned-lake",
        VersioningConfiguration={"Status": "Enabled"},
    )

    # Upload the same key twice — creates two versions
    s3.put_object(Bucket="versioned-lake", Key="config.json", Body=b'{"version": 1}')
    s3.put_object(Bucket="versioned-lake", Key="config.json", Body=b'{"version": 2}')

    # List all versions
    resp = s3.list_object_versions(Bucket="versioned-lake")
    for v in resp["Versions"]:
        print(f"  Key={v['Key']}  VersionId={v['VersionId'][:8]}...  IsLatest={v['IsLatest']}")

    # Read a specific version by ID
    old_version_id = [v["VersionId"] for v in resp["Versions"] if not v["IsLatest"]][0]
    old_body = s3.get_object(Bucket="versioned-lake", Key="config.json", VersionId=old_version_id)["Body"].read()
    print("Old version:", old_body)

  Key=config.json  VersionId=5c767f6d...  IsLatest=True
  Key=config.json  VersionId=082b8f35...  IsLatest=False
Old version: b'{"version": 1}'


In [12]:
# C2: Lifecycle rule — transition to Glacier after 30 days, expire after 365
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="lifecycle-demo")

    s3.put_bucket_lifecycle_configuration(
        Bucket="lifecycle-demo",
        LifecycleConfiguration={
            "Rules": [
                {
                    "ID": "archive-and-expire",
                    "Status": "Enabled",
                    "Filter": {"Prefix": "raw/"},
                    "Transitions": [
                        {"Days": 30, "StorageClass": "GLACIER"},
                    ],
                    "Expiration": {"Days": 365},
                }
            ]
        },
    )
    rules = s3.get_bucket_lifecycle_configuration(Bucket="lifecycle-demo")["Rules"]
    print("Lifecycle rule:", rules[0]["ID"], "— status:", rules[0]["Status"])

Lifecycle rule: archive-and-expire — status: Enabled


---
## Section D — Hive-Style Partition Layout

The most common S3 data lake pattern organizes data by `year=YYYY/month=MM/day=DD/`. This allows Athena and Glue to scan only the partitions you need (partition pruning), which reduces cost dramatically.

In [13]:
# D1: Upload partitioned data
import datetime

with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="hive-lake")

    # Simulate 3 days of data uploads
    base = datetime.date(2024, 1, 1)
    for i in range(3):
        day = base + datetime.timedelta(days=i)
        key = f"events/year={day.year}/month={day.month:02d}/day={day.day:02d}/data.parquet"
        df = pd.DataFrame({"event_id": [i * 100 + j for j in range(3)], "ts": str(day)})
        helpers.upload_df_as_parquet(s3, df, "hive-lake", key)
        print(f"Uploaded: {key}")

    # List only January 2024 data
    jan_objects = helpers.list_all_objects(s3, "hive-lake", prefix="events/year=2024/month=01/")
    print(f"\nJanuary partitions: {len(jan_objects)} file(s)")

Uploaded: events/year=2024/month=01/day=01/data.parquet
Uploaded: events/year=2024/month=01/day=02/data.parquet
Uploaded: events/year=2024/month=01/day=03/data.parquet

January partitions: 3 file(s)


In [14]:
# D2: Read a specific partition
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="read-partition")

    df = pd.DataFrame({"sensor_id": ["A", "B"], "reading": [42.1, 37.8]})
    helpers.upload_df_as_parquet(s3, df, "read-partition", "sensors/year=2024/month=01/day=15/data.parquet")

    key = "sensors/year=2024/month=01/day=15/data.parquet"
    df_read = helpers.download_df_from_parquet(s3, "read-partition", key)
    print(df_read)

  sensor_id  reading
0         A     42.1
1         B     37.8


## Section B — S3 Using boto3 Paginators (alternative to manual loop)

boto3 also has built-in `paginators` that handle the token loop automatically.

In [15]:
# Using a paginator instead of a manual continuation token loop
with mock_aws():
    s3 = helpers.make_boto3_client("s3")
    s3.create_bucket(Bucket="paginator-demo")
    for i in range(5):
        s3.put_object(Bucket="paginator-demo", Key=f"data/file_{i}.csv", Body=b"x")

    paginator = s3.get_paginator("list_objects_v2")
    all_keys = []
    for page in paginator.paginate(Bucket="paginator-demo", Prefix="data/"):
        all_keys.extend(obj["Key"] for obj in page.get("Contents", []))

    print("All keys:", all_keys)

All keys: ['data/file_0.csv', 'data/file_1.csv', 'data/file_2.csv', 'data/file_3.csv', 'data/file_4.csv']


## Summary

| API Call | Purpose |
|---|---|
| `create_bucket` | Create bucket (no location config for us-east-1) |
| `put_bucket_versioning` | Enable/suspend versioning |
| `put_bucket_tagging` | Cost allocation tags |
| `put_object` | Upload object (bytes) |
| `get_object` | Download — remember `.read()` on Body |
| `list_objects_v2` | List with prefix + pagination |
| `copy_object` | Server-side copy (no local download) |
| `delete_object` / `delete_objects` | Single / batch delete |
| `list_object_versions` | Show all versions of a key |
| `put_bucket_lifecycle_configuration` | Transition + expiry rules |

**Next notebook:** [02_glue_catalog_management.ipynb](02_glue_catalog_management.ipynb)